In [ ]:
%matplotlib notebook
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.nn import *

class EmbeddingAuto(EmbeddingBag):
    def __init__(s, n, k, l):
        super().__init__(n, k, mode='sum')
        s.l = Parameter(.5 ** torch.arange(l-1, -1., -1))
    def forward(s, x):
        return super().forward(x, per_sample_weights=s.l.expand_as(x))

class Log:
    n = 0
    def __init__(s, k=1, m=10):
        s.a = np.empty([2000,k], 'f')
        s.m = m
    def __call__(s, *x):
        s.a[s.n] = x
        s.n += 1
        if not s.n % s.m:
            s.draw()
    def draw(s):
        plt.cla()
        plt.plot(s.a[:s.n])
        plt.gcf().canvas.draw()

torch.set_default_tensor_type(torch.cuda.FloatTensor)
a = np.memmap('inews.B')
a = torch.tensor(a).as_strided([len(a)-4,5], [1,1])

In [ ]:
m = EmbeddingAuto(256, 256, 4)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, .01, len(a)//2048, pct_start=.01, div_factor=9, final_div_factor=99)
log = Log(m=1)
for t in tqdm(range(O.total_steps//100)):
    for t in range(100):
        t = a[torch.randint(len(a), [2048])]
        x = m(t[:,:-1].long())
        y = t[:,-1].long()
        functional.cross_entropy(x, y).backward()
        o.step()
        o.zero_grad()
        O.step()
    log((x.argmax(1)==y).sum().cpu()/len(y))
print(m.l/m.l.sum())

In [ ]:
m = Sequential(
    EmbeddingAuto(256, 1024, 4),
    Softmax(1),
    Linear(1024, 256),
)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, .01, len(a)*8//2048, pct_start=.01, div_factor=9, final_div_factor=999)
log = Log(m=1)
for t in tqdm(range(O.total_steps//100)):
    for t in range(100):
        t = a[torch.randint(len(a), [2048])]
        x = m(t[:,:-1].long())
        y = t[:,-1].long()
        functional.cross_entropy(x, y).backward()
        o.step()
        o.zero_grad()
        O.step()
    log((x.argmax(1)==y).sum().cpu()/len(y))
print(m[0].l/m[0].l.sum())

In [ ]:
m = Sequential(
    EmbeddingAuto(256, 1024, 4),
    LogSoftmax(1),
    Linear(1024, 256),
)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, .0002, len(a)*8//2048, pct_start=.01, div_factor=9, final_div_factor=999)
log = Log(m=1)
for t in tqdm(range(O.total_steps//100)):
    for t in range(100):
        t = a[torch.randint(len(a), [2048])]
        x = m(t[:,:-1].long())
        y = t[:,-1].long()
        functional.cross_entropy(x, y).backward()
        o.step()
        o.zero_grad()
        O.step()
    log((x.argmax(1)==y).sum().cpu()/len(y))
print(m[0].l/m[0].l.sum())

In [ ]:
m = Sequential(
    EmbeddingAuto(256, 1024, 4),
    ReLU(1),
    Linear(1024, 256),
)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, .001, len(a)*8//2048, pct_start=.1, div_factor=9, final_div_factor=99)
log = Log()
for t in tqdm(range(O.total_steps//100)):
    for t in range(100):
        t = a[torch.randint(len(a), [2048])]
        x = m(t[:,:-1].long())
        y = t[:,-1].long()
        functional.cross_entropy(x, y).backward()
        o.step()
        o.zero_grad()
        O.step()
    log((x.argmax(1)==y).sum().cpu()/len(y))
log.draw()
print(m[0].l/m[0].l.sum())
# ~20

In [ ]:
class Affine(Module):
    def __init__(s, k):
        super().__init__()
        s.w = Parameter(torch.ones(k))
        s.b = Parameter(torch.zeros(k))
    def forward(s, x):
        return s.b.addcmul(x, s.w)
m = Sequential(
    EmbeddingAuto(256, 1024, 4),
    Softmax(1),
    Affine(1024),
    ReLU(1),
    Linear(1024, 256),
)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, .001, len(a)*8//2048, pct_start=.1, div_factor=9, final_div_factor=99)
log = Log()
for t in tqdm(range(O.total_steps//100)):
    for t in range(100):
        t = a[torch.randint(len(a), [2048])]
        x = m(t[:,:-1].long())
        y = t[:,-1].long()
        functional.cross_entropy(x, y).backward()
        o.step()
        o.zero_grad()
        O.step()
    log((x.argmax(1)==y).sum().cpu()/len(y))
log.draw()
print(m[0].l/m[0].l.sum())